# Stored Procedures

# 1. Load Packages

In [1]:
import os
from dotenv import load_dotenv
import mysql.connector
import pandas as pd

# 2. Connect to MySQL Server

In [2]:
# Load .env
# load_dotenv()

# Force load
load_dotenv(dotenv_path=".env")   # force load

conn = mysql.connector.connect(
    user = os.getenv("MYSQL_USER"),
    host = os.getenv("MYSQL_HOST"),
    port = os.getenv("MYSQL_PORT"),
    password = os.getenv("MYSQL_PASSWORD"),
    database = "nth_highest_salary_int_db"
)
print("Connected to the MySQL server successfully")

Connected to the MySQL server successfully


# 3. Create a Cursor

In [3]:
cursor = conn.cursor()

# 4. nth Highist Salary

In [4]:
# Print the table

cursor.execute("SELECT * FROM `employees`")
results = cursor.fetchall()
headers = [col[0] for col in cursor.description]

display(pd.DataFrame(results, columns=headers))

,id,firstName,lastName,gender,salary
0,1,Ben,Hoskins,Male,70000
1,2,Mark,Hastings,Male,60000
2,3,Steve,Pound,Male,45000
3,4,Ben,Hoskins,Male,70000
4,5,Philip,Hastings,Male,45000
5,6,Mary,Lambeth,Female,30000
6,7,Valarie,Vikings,Female,35000
7,8,John,Stanmore,Male,80000


## 4.1 Using Derived Table + DISTINCT

In [5]:
# 4th highest salary
query = """
SELECT salary AS 4th_highest_salary FROM (SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT 4) D 
ORDER BY salary ASC LIMIT 1
"""

cursor.execute(query)
results = cursor.fetchall()
headers = [col[0] for col in cursor.description]

display(pd.DataFrame(results, columns=headers))

,4th_highest_salary
0,45000


In [6]:
query = """
CREATE PROCEDURE IF NOT EXISTS nth_highest_salary_proc (IN N INT) 
BEGIN
    SELECT salary AS nth_highest_salary FROM (SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT N) D 
    ORDER BY salary ASC LIMIT 1;
END
"""
cursor.execute(query)

In [7]:
# Call the procedure
cursor.callproc("nth_highest_salary_proc", (4, ))

for result in cursor.stored_results():
    rows = result.fetchall()
    headers = [col[0] for col in result.description]
    df = pd.DataFrame(rows, columns=headers)
    display(df)

C:\Users\adeet\AppData\Local\Temp\ipykernel_7796\2607081674.py:4: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():


,nth_highest_salary
0,45000


## 4.2 Using CTE + DISTINCT

In [8]:
query = """
CREATE PROCEDURE IF NOT EXISTS nth_highest_salary_cte_proc (IN N INT) 
BEGIN
    WITH cte AS (
        SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT N
    ) SELECT salary AS nth_highest_salary FROM cte ORDER BY salary ASC LIMIT 1;
END
"""
cursor.execute(query)

In [9]:
# Call the procedure
cursor.callproc("nth_highest_salary_cte_proc", (4, ))

for result in cursor.stored_results():
    rows = result.fetchall()
    headers = [col[0] for col in result.description]
    df = pd.DataFrame(rows, columns=headers)
    display(df)

C:\Users\adeet\AppData\Local\Temp\ipykernel_7796\184933844.py:4: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():


,nth_highest_salary
0,45000


## 4.3 Using Window Functions

In [10]:
query = """
CREATE PROCEDURE IF NOT EXISTS nth_highest_salary_window_proc (IN N INT) 
BEGIN
    SELECT salary AS nth_highest_salary 
    FROM (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM employees) D
    WHERE D.rnk = N LIMIT 1;
END
"""
cursor.execute(query)

In [11]:
# Call the procedure
cursor.callproc("nth_highest_salary_window_proc", (4, ))

for result in cursor.stored_results():
    rows = result.fetchall()
    headers = [col[0] for col in result.description]
    df = pd.DataFrame(rows, columns=headers)
    display(df)

C:\Users\adeet\AppData\Local\Temp\ipykernel_7796\350204928.py:4: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():


,nth_highest_salary
0,45000
